# Chebyshev-SNO: локальное обучение

Переносим A,n в тензорные узлы Chebyshev–Lobatto, добавляем cos/sin, нормированный k и x,y. DCT-I с корректными весами концов переводит значения в коэффициенты ряда. Нелинейные слои работают над retained коэффициентами, после чего обратный ряд даёт поле на Lobatto-сетке. Для loss возвращаем его в исходные FEM-узлы и считаем consistent комплексную FEM L2.

Это coefficient-space SNO; параметры и Helmholtz-задача отличаются от авторских экспериментов Fanaskov–Oseledets. [Первичная статья](https://arxiv.org/abs/2205.10573).

**Статичная задача:** геометрия и исходная среда №0 фиксированы; A,n и DG0-материалы одинаковы во всех примерах. Меняются k и θ. Один medium_id=0, разбиение по frequency_id: 670/170/160 частот (5360/1360/1280 полей). 1000 частот × 8 направлений = 8000 полей на полосу. Этот эксперимент проверяет интерполяцию параметров волны на одной среде.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'neural_operator').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from neural_operator.settings import (BASE, N_FREQUENCIES, N_DIRECTIONS, DATASET_TAG, POD_METHOD,
    LOW_DATA_DIR, HIGH_DATA_DIR, LOW_RUNS, HIGH_RUNS, SVD_RUNS, COMPARISON_RUNS)
DATA_DIR = LOW_DATA_DIR
RUNS = LOW_RUNS
import json
import numpy as np
import matplotlib.pyplot as plt
from neural_operator.data import load_dataset, write_dataset_info
from neural_operator.plotting import display_table, plot_prediction, plot_history


## Конфигурация и запуск

Первое выполнение обучает модель на локальном CPU. Повторное загружает только завершённый запуск с теми же данными и конфигурацией. `RETRAIN=True` запускает новое обучение; автоматического resume нет. Лучшая эпоха определяется по validation. Лимит эпох не означает сходимость. На каждой модели одинаковы split, seed и максимальный budget эпох, но числа параметров отличаются.


In [ ]:
from neural_operator.training import TrainConfig, run_experiment
METHOD = 'chebyshev'
RETRAIN = False
CONFIG = TrainConfig(method=METHOD, epochs=200, batch_size=32, width=24, depth=3,
                     rank=48, input_rank=32, resolution=32, modes=8, seed=42, threads=2, pod_method=POD_METHOD)
dataset = load_dataset(DATA_DIR, mmap=True)
OUT = RUNS / METHOD
predictor, metrics, history = run_experiment(dataset, OUT, CONFIG, retrain=RETRAIN)
print(json.dumps(metrics, indent=2, ensure_ascii=False))


In [ ]:
figure = plot_history(history)
figure.savefig(OUT / 'history.png', dpi=160)
plt.show()
display_table([{'method': METHOD, **metrics['test_fem_l2'], 'best_epoch': metrics['best_epoch'],
                'epochs': metrics['completed_epochs'], 'parameters': metrics['parameters']}])


In [ ]:
from neural_operator.diagnostics import chebyshev_reconstruction
val = dataset.split_indices()[1]
reconstruction = chebyshev_reconstruction(dataset, val, CONFIG.resolution, CONFIG.modes)
from neural_operator.training import summarize
print('Validation Chebyshev interpolation/truncation diagnostic:', summarize(dataset.relative_l2(dataset.u_scat[val], reconstruction)))


In [ ]:
test = dataset.split_indices()[2]
index = int(test[0])
prediction = predictor.predict_dataset(dataset, [index])[0]
figure = plot_prediction(dataset, index, prediction)
figure.savefig(OUT / 'prediction.png', dpi=160)
plt.show()
print('Self-contained checkpoint:', OUT / 'model.pt')


Модель, базисы, нормализация, геометрия и архитектура находятся в одном `model.pt`; история и метрики сохранены рядом. Отдельный [Inference.ipynb](Inference.ipynb) загружает этот файл в чистом ядре. Сопоставление всех методов: [Comparison.ipynb](Comparison.ipynb). Расширенный набор содержит 1000 частот × 8 направлений. Малый train loss не подтверждает перенос на новые среды; результаты старого пилота сохранены в docs/local_results.md.
